# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [1]:
#!pip install -qU ragas==0.2.10

In [2]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

fixes an issue that can happen with mac where you don't have tokenizer

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/terellbrown/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/terellbrown/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [4]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [6]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [7]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/terellbrown/terellcodes/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/terellbrown/terellcodes/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/terellbrown/terellcodes/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [11]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'ad0c43'. Skipping!
Property 'summary' already exists in node 'bfe004'. Skipping!
Property 'summary' already exists in node 'f41755'. Skipping!
Property 'summary' already exists in node '19685b'. Skipping!
Property 'summary' already exists in node '13a566'. Skipping!
Property 'summary' already exists in node '0f14bf'. Skipping!
Property 'summary' already exists in node '398562'. Skipping!
Property 'summary' already exists in node 'd2e2a3'. Skipping!
Property 'summary' already exists in node '08714f'. Skipping!
Property 'summary' already exists in node 'ceb47e'. Skipping!
Property 'summary' already exists in node '96fe1b'. Skipping!
Property 'summary' already exists in node 'b9223a'. Skipping!
Property 'summary' already exists in node '4f8d82'. Skipping!
Property 'summary' already exists in node '6c5b3a'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/41 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'ad0c43'. Skipping!
Property 'summary_embedding' already exists in node '19685b'. Skipping!
Property 'summary_embedding' already exists in node '13a566'. Skipping!
Property 'summary_embedding' already exists in node 'd2e2a3'. Skipping!
Property 'summary_embedding' already exists in node 'b9223a'. Skipping!
Property 'summary_embedding' already exists in node 'f41755'. Skipping!
Property 'summary_embedding' already exists in node 'bfe004'. Skipping!
Property 'summary_embedding' already exists in node '0f14bf'. Skipping!
Property 'summary_embedding' already exists in node '6c5b3a'. Skipping!
Property 'summary_embedding' already exists in node '96fe1b'. Skipping!
Property 'summary_embedding' already exists in node 'ceb47e'. Skipping!
Property 'summary_embedding' already exists in node '398562'. Skipping!
Property 'summary_embedding' already exists in node '08714f'. Skipping!
Property 'summary_embedding' already exists in node '4f8d82'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 39, relationships: 476)

We can save and load our knowledge graphs as follows.

In [12]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 39, relationships: 476)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

### ✅ ANSWER
#### Single Hop Specific Query Synthesizer:
This synthesizer generates straightforward, direct questions that can be answered using a single piece of information from a single document chunks in the knowledge base. These are typically fact-based or specific questions that don't require combining multiple pieces of information. (Example: "When was Bob Marley born?")

#### Multi Hop Abstract Query Synthesizer:
This synthesizer creates more complex questions that require synthesizing information from multiple document chunks to form abstract or high-level conclusions. It focuses on generating questions that test the system's ability to understand broader concepts and relationships between different pieces of information. Focuses less on specific properties shared between document chunks and more on conceptual relationships (likely matches on embedding). (Example: "What technological and economic breakthroughs led to the mass adoption of LLM?")

#### Multi Hop Specific Query Synthesizer:
While this does use multiple question-answer pairs like the abstract synthesizer, it differs in that it creates specific, detailed questions that require connecting multiple concrete facts from multiple document chunks rather than abstract concepts. These questions test the system's ability to combine specific pieces of information from different sources to provide a precise answer. (Example: "What is the relationship between {concept from chunk A} and {concept from chunk B}?")


Finally, we can use our `TestSetGenerator` to generate our testset!

In [41]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Volume 2 in the c...,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not specify the cont...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) say about the academ...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,What is Volume 8 about?,[Inclusion of Clinical Work in a Standard Term...,The context references Volume 8 in relation to...,single_hop_specifc_query_synthesizer
3,FWS is not like other programs because it does...,[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
4,How does the Pell Grannt affect disbursement t...,[both the credit or clock hours and the weeks ...,The Pell Grant amount a student is eligible to...,single_hop_specifc_query_synthesizer
5,How do the requirements outlined in 34 CFR 668...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The requirements in 34 CFR 668.3(a) specify th...,multi_hop_abstract_query_synthesizer
6,How do the disbursement timing requirements fo...,[<1-hop>\n\nboth the credit or clock hours and...,Disbursement timing requirements for federal f...,multi_hop_abstract_query_synthesizer
7,what program reqirements are in 34 CFR 668.3(b...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",34 CFR 668.3(b) specifies the weeks of instruc...,multi_hop_abstract_query_synthesizer
8,How do Chapters 2 and 3 collectively address t...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 details the definitions and requirem...,multi_hop_specific_query_synthesizer
9,How do Appendix A and Appendix B relate to dis...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,Appendix B provides detailed guidance on disbu...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '264dcd'. Skipping!
Property 'summary' already exists in node 'd3e617'. Skipping!
Property 'summary' already exists in node '902f76'. Skipping!
Property 'summary' already exists in node '61cbf0'. Skipping!
Property 'summary' already exists in node '6db158'. Skipping!
Property 'summary' already exists in node '5798d9'. Skipping!
Property 'summary' already exists in node '04802e'. Skipping!
Property 'summary' already exists in node '316d9a'. Skipping!
Property 'summary' already exists in node '834116'. Skipping!
Property 'summary' already exists in node 'ecd5e9'. Skipping!
Property 'summary' already exists in node 'aa5166'. Skipping!
Property 'summary' already exists in node 'a940b7'. Skipping!
Property 'summary' already exists in node 'b0d06d'. Skipping!
Property 'summary' already exists in node '0df6d3'. Skipping!


...

Property 'summary_embedding' already exists in node '264dcd'. Skipping!
Property 'summary_embedding' already exists in node '902f76'. Skipping!
Property 'summary_embedding' already exists in node '316d9a'. Skipping!
Property 'summary_embedding' already exists in node '61cbf0'. Skipping!
Property 'summary_embedding' already exists in node '6db158'. Skipping!
Property 'summary_embedding' already exists in node 'd3e617'. Skipping!
Property 'summary_embedding' already exists in node '834116'. Skipping!
Property 'summary_embedding' already exists in node '5798d9'. Skipping!
Property 'summary_embedding' already exists in node 'a940b7'. Skipping!
Property 'summary_embedding' already exists in node '0df6d3'. Skipping!
Property 'summary_embedding' already exists in node 'b0d06d'. Skipping!
Property 'summary_embedding' already exists in node '04802e'. Skipping!
Property 'summary_embedding' already exists in node 'ecd5e9'. Skipping!
Property 'summary_embedding' already exists in node 'aa5166'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [42]:
import pandas as pd
pd.set_option('display.max_colwidth', None)  # Show full column content
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Auto-detect display width

In [43]:
dataset.to_pandas()

user_input  \
0                                                                                                                                                                                                                                                               What is the role of the School Participation Division?   
1                                                                                                                                                                                                                                                                                       What is 34 CFR 668.3(a) about?   
2                                                                                                                                                                                                                                           What is Volume 8 about in the context of clinical work and standard terms?   
3                                                                           Could you explain the specific circumstances under which the Federal Work-Study (FWS) program is exempt from the requirement of disbursing payments on a payment period basis, and how this distinguishes it from other Title IV programs?   
4   How do the regulatory citations, specifically 34 CFR 668.3(a) and 34 CFR 668.3(b), support the requirement for separate program versions and enrollment considerations when defining academic years for different programs, and how do these regulations ensure compliance with federal financial aid regulations?   
5                                                                                 How does the guidance on determining completion of academic milestones relate to initial disbursement requirements for federal student aid programs, especially considering accelerated progression and subscription-based programs?   
6                                                                                                                                                                                                                      Wht is the minmum instrucional weeks for credit and clock hour programs as per 34 CFR 668.3(b)?   
7                                                                                                                                                                                           How do the program requirements outlined in 34 CFR 668.3(b) relate to the academic year definitions for Title IV purposes?   
8                                                                                                                                                                                                                                      Chapter 2 and Chapter 3 how do they relate in academic programs and compliance?   
9                                                                              How does the inclusion of clinical work in standard term periods, as described in Volume 8, impact the calculation of federal financial aid disbursements and loan limits for students enrolled in nonstandard or accelerated programs?   
10                                                                                                                                    how Appendix A and B tell about disbursement timing in subscription programs and credit hours and weeks for Pell Grant and Direct Loan, and how does that matter for compliance?   
11                                                                                     How do Volume 2 and Volume 7 relate to the academic year requirements and disbursement rules for clock-hour and credit-hour programs, especially when considering the impact of accelerated progression on disbursement timing?   

                                                                                                                                                                                                                         

We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [18]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [19]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [20]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [23]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [25]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [28]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct PLUS Loan or student Federal PLUS Loan  \n- Subsidized Federal Stafford Loans  \n- Unsubsidized Federal Stafford Loans  \n- Federal SLS Loans  \n- Federal PLUS Loans  \n- Direct Subsidized Loan  \n- Direct Unsubsidized Loan  \n- Direct Consolidation Loan  \n- Federal Consolidation Loan  \n\nAdditionally, Direct Subsidized Loans are available only to undergraduate students, and graduate or professional students are eligible for Direct Unsubsidized Loans but not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [29]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [30]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

### ✅ ANSWER
- `qa_evaluator`: based on the judgment of the LLM does the response accurately answer the question
- `labeled_helpfulness_evaluator`: Compares the RAG app output to the reference answer to determine if the output accurately answer the question
- `empathy_evaluator`: does the application respond with a level of empathy that matches the level of inconvenience and fustration that the user is experiencing

## LangSmith Evaluation

In [31]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'respectful-milk-14' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/38dc4fa4-f8ff-4c5e-9f23-d649a1883f00/compare?selectedSessions=3a32aa04-0fb2-4aec-8635-27d6572eec16




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 7 relate to the aca...,"Based on the provided context, Volume 2 addres...",None,Volume 2 discusses the academic year requireme...,1,1,0,7.227043,c8228420-a33d-47cd-98fd-a29d64bb7c00,d15137ec-d859-4903-ae65-ac4ec461c0ee
1,how Appendix A and B tell about disbursement t...,Based on the provided context:\n\n**Appendix B...,None,Appendix B gives detailed guidance on disburse...,1,1,0,5.795291,65a9abda-8db5-4b85-937e-ebbbe8e9bdac,2b6906ba-2218-4ead-940f-c668a2bf368f
2,How does the inclusion of clinical work in sta...,"Based on the provided context, the inclusion o...",None,"According to Volume 8, clinical work conducted...",1,1,0,7.851044,add06a13-45e4-4d71-b821-d6adb40fb926,de2c0057-cc39-4b3c-ad90-172b53eff2e9
3,Chapter 2 and Chapter 3 how do they relate in ...,I don't know.,None,Chapter 2 discusses the inclusion of clinical ...,0,0,0,0.966764,c72488c4-624c-4536-beab-f484d59b1751,3cda8eff-4fba-4e87-8747-a6472d884511
4,How do the program requirements outlined in 34...,"Based on the provided context, the program req...",None,The program requirements outlined in 34 CFR 66...,1,0,0,3.028810,4f9300fa-8c73-4e56-a48d-0f944b063e07,f75e40bd-8d1c-4c30-882c-7b14f56da266
5,Wht is the minmum instrucional weeks for credi...,The minimum instructional weeks as per 34 CFR ...,None,"According to 34 CFR 668.3(b), the minimum inst...",1,1,0,1.675352,840927d6-d443-4d73-8087-975bcbfaf699,9ee9b1b6-8832-488e-8994-f42bdc9b3c8e
6,How does the guidance on determining completio...,"Based on the provided context, the guidance on...",None,The guidance on determining completion of acad...,1,1,0,6.195669,043d593b-ac31-4524-818d-7b7af7b51244,5e30b795-6ff9-4f4a-a85a-f4b9dafc67b2
7,"How do the regulatory citations, specifically ...",I don't know.,None,The regulatory citations 34 CFR 668.3(a) and 3...,0,0,0,0.857949,18ff7b42-be51-4667-b228-85fb98c58495,ec7517a8-9feb-406c-a6e0-cb3733b6277c
8,Could you explain the specific circumstances u...,Based on the provided context:\n\nThe Federal ...,None,The payment period is applicable to all Title ...,1,1,0,3.898776,44b1fe4c-6491-43e9-9011-95c875fe6b77,4d422357-caca-440d-84fd-a65f87a542bf
9,What is Volume 8 about in the context of clini...,Volume 8 discusses exceptions to the normal lo...,None,"Volume 8, Chapter 3 provides guidance on inclu...",1,1,0,2.291015,b036fbf7-ecbb-4be4-8227-a9bfe4a436db,a42301b6-93f4-4959-a966-8c936a403ad3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [32]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [33]:
rag_documents = docs

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

if we know the chunking process we are using does not cut off context (only produces paragraphs for example) then no need for overlap

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### ✅ ANSWER
Increasing the chunk size means more context and semantic meaning is captured in each chunk which leads to:
- when chunks are returned there is more context available to help you answer the question which could lead to better answers the question
- when applying similarity search it may be difficult to match a very specific to the appropriate chunk because the part of the chunk that relates to the query has less influence on the semanting meaning of the chunk

Increasing the chunk size means less chunks are created which leads to:
- fewer chunks that need to be compared during similarity search and lower latency
- less storage capacity consumed

In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### ✅ ANSWER
A larger embedding model, means more semantic meaning can be captured in the generated vector which leads to:
- a chunk can be matched to a query with greater semantic granularity => improved retrieval accuracy => improved accuracy of answer
- consumes more time and resources to generate embeddings and perform semantic search => increased latency and cost => worst user experience, less profit

In [36]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided in the context, there are several kinds of loans available to students and their parents to help cover the Cost of Attendance (COA):\n\n1. **Direct Subsidized Loans:** These loans are based on financial need and the federal government pays the interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans:** These are not based on financial need, and interest accrues while the student is in school. Both dependent and independent students may qualify.\n\n3. **Direct PLUS Loans:** These loans are available to parents of dependent students (Direct PLUS Loans for parents) or to graduate/professional students (student Direct PLUS Loans). There is no fixed loan limit for PLUS Loans, but they cannot exceed the student's COA minus other financial aid.\n\nI hope this helps clarify the options available for financing education. If you have more questions or need further explanation, please feel free to

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'complicated-vessel-1' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/38dc4fa4-f8ff-4c5e-9f23-d649a1883f00/compare?selectedSessions=44c24355-881e-44b7-9e47-960173430ff2




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 7 relate to the aca...,Thank you so much for your thoughtful question...,None,Volume 2 discusses the academic year requireme...,0,0,1,5.234841,c8228420-a33d-47cd-98fd-a29d64bb7c00,e61dbef3-b13f-4292-8be9-b34b354986a5
1,how Appendix A and B tell about disbursement t...,Thank you for your thoughtful question. It sou...,None,Appendix B gives detailed guidance on disburse...,1,1,1,21.343082,65a9abda-8db5-4b85-937e-ebbbe8e9bdac,f6428e08-9ef0-4062-94f3-900a0da1dcc9
2,How does the inclusion of clinical work in sta...,Thank you for your thoughtful question. Based ...,None,"According to Volume 8, clinical work conducted...",1,0,1,5.626362,add06a13-45e4-4d71-b821-d6adb40fb926,896429d9-e996-49d7-a71a-1bf7856fd5e4
3,Chapter 2 and Chapter 3 how do they relate in ...,Thank you for your thoughtful question about h...,None,Chapter 2 discusses the inclusion of clinical ...,0,0,1,5.776588,c72488c4-624c-4536-beab-f484d59b1751,8b18463e-acf1-4f2b-b3f1-62a6d6eeb18c
4,How do the program requirements outlined in 34...,Thank you for your thoughtful question. Based ...,None,The program requirements outlined in 34 CFR 66...,1,0,1,4.407723,4f9300fa-8c73-4e56-a48d-0f944b063e07,e9ab19c7-21d7-4fd3-a92e-30cffc17b4cf
5,Wht is the minmum instrucional weeks for credi...,Thank you for your thoughtful question. Based ...,None,"According to 34 CFR 668.3(b), the minimum inst...",0,0,1,4.842763,840927d6-d443-4d73-8087-975bcbfaf699,e0cc9308-4672-4ab5-939e-c7def01a0721
6,How does the guidance on determining completio...,Thank you for your thoughtful question. From t...,None,The guidance on determining completion of acad...,1,1,1,6.184538,043d593b-ac31-4524-818d-7b7af7b51244,e1ec7681-f93d-4709-900d-eb4a02425c67
7,"How do the regulatory citations, specifically ...",Thank you for your thoughtful question. I unde...,None,The regulatory citations 34 CFR 668.3(a) and 3...,0,0,1,7.476423,18ff7b42-be51-4667-b228-85fb98c58495,94022aac-fb87-4ea1-80b8-9215f4bb329c
8,Could you explain the specific circumstances u...,Thank you for your thoughtful question—it's im...,None,The payment period is applicable to all Title ...,1,1,1,9.726879,44b1fe4c-6491-43e9-9011-95c875fe6b77,9ade9e97-6a00-4c0d-9f65-f0e1310d295b
9,What is Volume 8 about in the context of clini...,Thank you for your question. From the context ...,None,"Volume 8, Chapter 3 provides guidance on inclu...",1,1,1,3.930123,b036fbf7-ecbb-4be4-8227-a9bfe4a436db,3f943a1e-5135-4126-b4b3-f531a0dfe918


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

### ✅ ANSWER


![Experiment comparison showing metrics for correctness, empathy, and helpfulness between two RAG implementations](experiment_comparison.png)







### ✅ ANSWER
The evaluation results showed:
1. The empathy-focused chain scored higher on empathy metrics (all answers became empathetic)
2. However, it had fewer correct answers compared to the basic chain
3. Both chains maintained similar helpfulness scores

This suggests that while we successfully improved the emotional intelligence of the responses, it came at a slight cost to factual accuracy. This is a common trade-off in RAG systems where adding additional objectives (like empathy) can sometimes reduce performance on the primary objective (accuracy).